In [2]:
import rich
import logging
import glob

import matplotlib.pyplot as plt
import time
import os
import awkward as ak
import numpy as np
import pandas as pd
from lgdo import lh5
from legendmeta import LegendMetadata
from dbetto import Props
from pygama.pargen.AoE_cal import *
from pygama.pargen.AoE_cal import CalAoE, Pol1, SigmaFit, aoe_peak
from pygama.pargen.data_cleaning import get_tcm_pulser_ids
from pygama.pargen.utils import load_data
from tqdm import tqdm

%matplotlib inline

logging.basicConfig(level=logging.INFO)
logging.getLogger('numba').setLevel(logging.INFO)
logging.getLogger('parse').setLevel(logging.INFO)

# Function

In [22]:
def single_plot(data, par, title = "xxx", cmax = 6):
    classifier = data[f"classifier_{par}"]

    m_all = ~data["is_discharge"]
    m_pl = m_all & data["is_pulser"]
    m_ft = m_all & data["forced_trigger"]
    m_clean = m_all & ~data["forced_trigger"] & ~data["is_pulser"] & data["is_below_500keV"]

    cmin = -6
    bins = 250

    plt.hist(classifier[m_all], bins=bins, range=(cmin, cmax), histtype='step', label='All')
    plt.hist(classifier[m_pl], bins=bins, range=(cmin, cmax), histtype='step', label='PL')
    plt.hist(classifier[m_ft], bins=bins, range=(cmin, cmax), histtype='step', label='FT')
    plt.hist(classifier[m_clean], bins=bins, range=(cmin, cmax), histtype='step', label='~PL & ~FT & <500 keV')


    plt.axvspan(-25, -5, color = 'grey', alpha=0.15)
    plt.axvspan(5,cmax, color = 'grey', alpha=0.15)
    plt.vlines(-5, 0, 1e7, ls = '--', color = 'grey' )
    plt.vlines(5, 0, 1e7, ls = '--', color = 'grey', label = 'Classifier CUT' )
    plt.ylabel('Counts')
    plt.title(f"{title} : r000 + r001")
    plt.xlabel(f'Classifier {par}')
    plt.yscale('log')
    plt.xlim(cmin, cmax)
    plt.legend(loc = 'upper left')

    return

In [30]:
def energy_spectrum_cut_plot(data, classifier_par):
    ene = data['cuspEmax_ctc_cal']
    
    m_discharge = ~data["is_discharge"]
    m_pulser = data["is_pulser"]
    m_ft = data["forced_trigger"]
    m_lowE = data["is_below_500keV"]
    m_fix = data["is_HPGe"]&~data["is_muon"]
    m_classifier = data[f"is_valid_{classifier_par}"]
    
    
    CUT = len(ene[m_discharge & ~m_pulser & ~m_ft & m_fix & ~m_classifier]) * 100/ len(ene[m_discharge & ~m_pulser & ~m_ft & m_fix])
    PASS = len(ene[m_discharge & ~m_pulser & ~m_ft & m_fix & m_classifier]) * 100/ len(ene[m_discharge & ~m_pulser & ~m_ft & m_fix])
    
    emin = -1000
    emax = 6000
    bins = int((emax-emin)/10)
    
    
    plt.figure(figsize = (10,3))
    plt.hist(ene[m_discharge & ~m_pulser & ~m_ft & m_fix], 
             bins = bins, 
             range=(emin, emax), 
             histtype = 'step',
             color = 'gray',
             label = 'All'
            )
    
    plt.hist(ene[m_discharge & ~m_pulser & ~m_ft & m_fix & m_classifier], 
             bins = bins, 
             range=(emin, emax), 
             histtype = 'step',
             color = 'deepskyblue',
             label = f'is_valid_{classifier_par}, PASS={PASS:.3f}%'
            )
    
    plt.hist(ene[m_discharge & ~m_pulser & ~m_ft & m_fix & ~m_classifier], 
             bins = bins, 
             range=(emin, emax), 
             histtype = 'step',
             color = 'brown',
             label = f'~is_valid_{classifier_par}, CUT={CUT:.3f}%'
            )
    
    plt.legend(loc = 'upper right')
    plt.xlabel('Energy [keV]')
    plt.ylabel('Counts / 10 keV')
    plt.title(f'{classifier_par} cut')
    plt.xlim(emin, emax)
    plt.yscale('log')

    

# Get data

In [26]:
from pathlib import Path
import pandas as pd

folders = ["Parquet data r000", "Parquet data r001"]

def load_detector(filename):
    return pd.concat(
        [pd.read_parquet(Path(folder) / filename) for folder in folders],
        ignore_index=True
    )

df_coax = load_detector("data_Coax_phy_ALL.parquet")
df_bege = load_detector("data_BEGe_phy_ALL.parquet")
df_ppc  = load_detector("data_PPC_phy_ALL.parquet")
df_icpc = load_detector("data_ICPC_phy_ALL.parquet")

df_all = pd.concat(
    [df_coax, df_bege, df_ppc, df_icpc],
    ignore_index=True
)

In [6]:
params = ["bl_slope", 
          "bl_slope_rms",
         "tail_rms"]


# bl_slope

# bl_slope_rms

# tail_rms

### PLot energy

In [32]:
'''
energy_spectrum_cut_plot(df_all, "bl_slope")
energy_spectrum_cut_plot(df_all, "bl_slope_rms")
energy_spectrum_cut_plot(df_all, "tail_rms")'''

'\nenergy_spectrum_cut_plot(df_all, "bl_slope")\nenergy_spectrum_cut_plot(df_all, "bl_slope_rms")\nenergy_spectrum_cut_plot(df_all, "tail_rms")'